# Notebook de gráficos para `security.csv`

Este notebook analiza un dataset de **logs de seguridad de Windows** y genera gráficos para exploración, detección y análisis de eventos sospechosos.

El notebook hace dos cosas en cada gráfico:

1. **Muestra el gráfico** dentro del notebook.
2. **Guarda el gráfico** automáticamente en el directorio `graficos`.

> Archivo esperado en el mismo directorio del notebook: `security(1).csv`

In [ ]:
# ============================================================
# 01. IMPORTACIÓN DE LIBRERÍAS
# ============================================================

# pandas permite cargar, limpiar, transformar y analizar datos tabulares.
import pandas as pd

# matplotlib permite crear gráficos estáticos y guardarlos como imágenes.
import matplotlib.pyplot as plt

# pathlib permite trabajar con rutas de archivos y carpetas de forma segura.
from pathlib import Path

# numpy se usará para algunas operaciones numéricas auxiliares.
import numpy as np

# warnings se usa para ocultar advertencias que no afectan el análisis.
import warnings
warnings.filterwarnings('ignore')

# Configuración general para que los gráficos se vean mejor en el notebook.
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['font.size'] = 10


In [ ]:
# ============================================================
# 02. CONFIGURACIÓN DE RUTAS
# ============================================================

# Nombre del archivo CSV de entrada.
# Debe estar ubicado en el mismo directorio donde se ejecuta este notebook.
ARCHIVO_CSV = 'security.csv'

# Directorio donde se guardarán todos los gráficos generados.
DIR_GRAFICOS = Path('graficos')

# Crear la carpeta si no existe.
DIR_GRAFICOS.mkdir(exist_ok=True)

print(f'Archivo de entrada: {ARCHIVO_CSV}')
print(f'Directorio de gráficos: {DIR_GRAFICOS.resolve()}')


In [ ]:
# ============================================================
# 03. CARGA DEL DATASET
# ============================================================

# Cargar el archivo CSV en un DataFrame de pandas.
df = pd.read_csv(ARCHIVO_CSV)

# Mostrar dimensiones del dataset: filas y columnas.
print('Dimensiones del dataset:', df.shape)

# Mostrar las primeras filas para verificar la estructura.
df.head()


In [ ]:
# ============================================================
# 04. REVISIÓN INICIAL DE COLUMNAS Y TIPOS DE DATOS
# ============================================================

print('Columnas disponibles:')
for col in df.columns:
    print('-', col)

print('\nTipos de datos:')
print(df.dtypes)

print('\nValores nulos por columna:')
print(df.isna().sum())


In [ ]:
# ============================================================
# 05. PREPARACIÓN DE DATOS
# ============================================================

# Convertir la columna timestamp a tipo fecha/hora.
# errors='coerce' convierte valores inválidos en NaT, evitando que el notebook falle.
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

# Crear columnas derivadas útiles para gráficos temporales.
df['fecha'] = df['timestamp'].dt.date
df['hora'] = df['timestamp'].dt.hour
df['dia_semana'] = df['timestamp'].dt.day_name()
df['minuto'] = df['timestamp'].dt.floor('min')
df['segundo'] = df['timestamp'].dt.floor('s')

# Crear una versión textual de la etiqueta sospechosa.
df['estado_seguridad'] = df['es_sospechoso'].map({0: 'Normal', 1: 'Sospechoso'}).fillna(df['es_sospechoso'].astype(str))

# Orden recomendado de días de semana para gráficos.
orden_dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Vista general posterior a la preparación.
df.head()


In [ ]:
# ============================================================
# 06. FUNCIÓN GENERAL PARA GRAFICAR, MOSTRAR Y GUARDAR
# ============================================================

def limpiar_nombre_archivo(texto):
    """
    Convierte el título de un gráfico en un nombre de archivo seguro.
    Reemplaza espacios y caracteres especiales por guiones bajos.
    """
    texto = str(texto).strip().lower()
    caracteres_invalidos = ['/', '\\', ':', '*', '?', '"', '<', '>', '|', '(', ')', '[', ']', '{', '}', ',', ';']
    for c in caracteres_invalidos:
        texto = texto.replace(c, '')
    texto = texto.replace(' ', '_')
    texto = texto.replace('__', '_')
    return texto[:120]


def guardar_y_mostrar(titulo):
    """
    Guarda el gráfico actual en la carpeta graficos y luego lo muestra en el notebook.
    """
    nombre_archivo = limpiar_nombre_archivo(titulo) + '.png'
    ruta = DIR_GRAFICOS / nombre_archivo
    plt.title(titulo)
    plt.tight_layout()
    plt.savefig(ruta, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Gráfico guardado en: {ruta}')


def grafico_barras(serie, titulo, xlabel='', ylabel='Cantidad', rotacion=45, top=None):
    """
    Crea un gráfico de barras a partir de una Serie de pandas.
    Permite limitar a los Top N valores más frecuentes.
    """
    datos = serie.copy()
    if top is not None:
        datos = datos.head(top)
    plt.figure(figsize=(12, 6))
    datos.plot(kind='bar')
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(rotation=rotacion, ha='right')
    guardar_y_mostrar(titulo)


def grafico_pie(serie, titulo):
    """
    Crea un gráfico circular a partir de una Serie de pandas.
    """
    plt.figure(figsize=(8, 8))
    serie.plot(kind='pie', autopct='%1.1f%%', startangle=90)
    plt.ylabel('')
    guardar_y_mostrar(titulo)


## 1. Gráficos generales del dataset

In [ ]:
# ============================================================
# 07. GRÁFICOS GENERALES
# ============================================================

grafico_barras(df['estado_seguridad'].value_counts(),
               'Eventos normales vs eventos sospechosos',
               xlabel='Estado de seguridad', rotacion=0)

grafico_pie(df['estado_seguridad'].value_counts(),
            'Porcentaje de eventos normales vs sospechosos')

grafico_barras(df['tipo_evento'].value_counts(),
               'Distribución por tipo de evento',
               xlabel='Tipo de evento', rotacion=0)

grafico_pie(df['tipo_evento'].value_counts(),
            'Porcentaje por tipo de evento')

grafico_barras(df['event_id'].value_counts().sort_index(),
               'Distribución de Event ID de Windows',
               xlabel='Event ID', rotacion=0)

grafico_barras(df['provider'].value_counts(),
               'Distribución por proveedor del log',
               xlabel='Provider', rotacion=45)


## 2. Gráficos por equipos, usuarios, dominios e IPs

In [ ]:
# ============================================================
# 08. EQUIPOS, USUARIOS, DOMINIOS E IPS
# ============================================================

grafico_barras(df['host'].value_counts(),
               'Eventos por host',
               xlabel='Host', rotacion=45)

grafico_barras(df['domain'].value_counts(),
               'Eventos por dominio',
               xlabel='Dominio', rotacion=0)

grafico_barras(df['user'].value_counts(),
               'Eventos por usuario',
               xlabel='Usuario', rotacion=45)

grafico_barras(df['user'].value_counts(),
               'Top 20 usuarios con más eventos',
               xlabel='Usuario', rotacion=45, top=20)

grafico_barras(df['source_ip'].value_counts(),
               'Top 20 IPs de origen con más eventos',
               xlabel='IP de origen', rotacion=45, top=20)

# IPs de origen asociadas a eventos sospechosos.
df_sospechoso = df[df['es_sospechoso'] == 1]
if not df_sospechoso.empty:
    grafico_barras(df_sospechoso['source_ip'].value_counts(),
                   'Top 20 IPs de origen con más eventos sospechosos',
                   xlabel='IP de origen', rotacion=45, top=20)
else:
    print('No se encontraron eventos sospechosos para graficar IPs sospechosas.')


## 3. Gráficos por procesos, logon type y puertos

In [ ]:
# ============================================================
# 09. PROCESOS, LOGON TYPE Y PUERTOS
# ============================================================

grafico_barras(df['process'].value_counts(),
               'Eventos por proceso',
               xlabel='Proceso', rotacion=45)

grafico_barras(df['process'].value_counts(),
               'Top 20 procesos con más eventos',
               xlabel='Proceso', rotacion=45, top=20)

grafico_barras(df['logon_type'].value_counts().sort_index(),
               'Distribución por Logon Type',
               xlabel='Logon Type', rotacion=0)

if not df_sospechoso.empty:
    grafico_barras(df_sospechoso['process'].value_counts(),
                   'Procesos más frecuentes en eventos sospechosos',
                   xlabel='Proceso', rotacion=45, top=20)
    
    grafico_barras(df_sospechoso['logon_type'].value_counts().sort_index(),
                   'Logon Type en eventos sospechosos',
                   xlabel='Logon Type', rotacion=0)

# Histograma de puertos de origen.
plt.figure(figsize=(12, 6))
df['source_port'].dropna().plot(kind='hist', bins=50)
plt.xlabel('Puerto de origen')
plt.ylabel('Frecuencia')
guardar_y_mostrar('Histograma de puertos de origen')

# Top puertos de origen.
grafico_barras(df['source_port'].value_counts(),
               'Top 20 puertos de origen más frecuentes',
               xlabel='Puerto de origen', rotacion=45, top=20)


## 4. Gráficos temporales

In [ ]:
# ============================================================
# 10. ANÁLISIS TEMPORAL
# ============================================================

# Eventos por día.
eventos_dia = df.groupby('fecha').size()
plt.figure(figsize=(12, 6))
eventos_dia.plot(kind='line', marker='o')
plt.xlabel('Fecha')
plt.ylabel('Cantidad de eventos')
guardar_y_mostrar('Cantidad de eventos por día')

# Eventos sospechosos por día.
if not df_sospechoso.empty:
    sospechosos_dia = df_sospechoso.groupby('fecha').size()
    plt.figure(figsize=(12, 6))
    sospechosos_dia.plot(kind='line', marker='o')
    plt.xlabel('Fecha')
    plt.ylabel('Cantidad de eventos sospechosos')
    guardar_y_mostrar('Cantidad de eventos sospechosos por día')

# Eventos por hora del día.
grafico_barras(df['hora'].value_counts().sort_index(),
               'Cantidad de eventos por hora del día',
               xlabel='Hora', rotacion=0)

# Eventos sospechosos por hora del día.
if not df_sospechoso.empty:
    grafico_barras(df_sospechoso['hora'].value_counts().sort_index(),
                   'Cantidad de eventos sospechosos por hora del día',
                   xlabel='Hora', rotacion=0)

# Eventos por minuto.
eventos_minuto = df.groupby('minuto').size()
plt.figure(figsize=(14, 6))
eventos_minuto.plot(kind='line')
plt.xlabel('Minuto')
plt.ylabel('Cantidad de eventos')
guardar_y_mostrar('Eventos por minuto')

# Eventos sospechosos por minuto.
if not df_sospechoso.empty:
    sospechosos_minuto = df_sospechoso.groupby('minuto').size()
    plt.figure(figsize=(14, 6))
    sospechosos_minuto.plot(kind='line')
    plt.xlabel('Minuto')
    plt.ylabel('Cantidad de eventos sospechosos')
    guardar_y_mostrar('Eventos sospechosos por minuto')


## 5. Cruces de variables para análisis de seguridad

In [ ]:
# ============================================================
# 11. CRUCES DE VARIABLES
# ============================================================

# Event ID vs estado de seguridad.
tabla_event_estado = pd.crosstab(df['event_id'], df['estado_seguridad'])
tabla_event_estado.plot(kind='bar', stacked=True, figsize=(12, 6))
plt.xlabel('Event ID')
plt.ylabel('Cantidad')
plt.xticks(rotation=0)
guardar_y_mostrar('Event ID por estado de seguridad')

# Logon type vs estado de seguridad.
tabla_logon_estado = pd.crosstab(df['logon_type'], df['estado_seguridad'])
tabla_logon_estado.plot(kind='bar', stacked=True, figsize=(12, 6))
plt.xlabel('Logon Type')
plt.ylabel('Cantidad')
plt.xticks(rotation=0)
guardar_y_mostrar('Logon Type por estado de seguridad')

# Usuario vs estado de seguridad para los top 20 usuarios.
top_usuarios = df['user'].value_counts().head(20).index
tabla_usuario_estado = pd.crosstab(df[df['user'].isin(top_usuarios)]['user'],
                                   df[df['user'].isin(top_usuarios)]['estado_seguridad'])
tabla_usuario_estado.plot(kind='bar', stacked=True, figsize=(14, 6))
plt.xlabel('Usuario')
plt.ylabel('Cantidad')
plt.xticks(rotation=45, ha='right')
guardar_y_mostrar('Top 20 usuarios por estado de seguridad')

# Host vs estado de seguridad.
tabla_host_estado = pd.crosstab(df['host'], df['estado_seguridad'])
tabla_host_estado.plot(kind='bar', stacked=True, figsize=(14, 6))
plt.xlabel('Host')
plt.ylabel('Cantidad')
plt.xticks(rotation=45, ha='right')
guardar_y_mostrar('Host por estado de seguridad')

# Proceso vs estado de seguridad para top 20 procesos.
top_procesos = df['process'].value_counts().head(20).index
tabla_proceso_estado = pd.crosstab(df[df['process'].isin(top_procesos)]['process'],
                                   df[df['process'].isin(top_procesos)]['estado_seguridad'])
tabla_proceso_estado.plot(kind='bar', stacked=True, figsize=(14, 6))
plt.xlabel('Proceso')
plt.ylabel('Cantidad')
plt.xticks(rotation=45, ha='right')
guardar_y_mostrar('Top 20 procesos por estado de seguridad')


## 6. Mapas de calor

In [ ]:
# ============================================================
# 12. HEATMAPS CON MATPLOTLIB
# ============================================================

def graficar_heatmap(tabla, titulo, xlabel='', ylabel=''):
    """
    Genera un mapa de calor usando únicamente matplotlib.
    """
    plt.figure(figsize=(12, 7))
    plt.imshow(tabla.values, aspect='auto')
    plt.colorbar(label='Cantidad')
    plt.xticks(range(len(tabla.columns)), tabla.columns, rotation=45, ha='right')
    plt.yticks(range(len(tabla.index)), tabla.index)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    
    # Colocar valores dentro de cada celda si la tabla no es demasiado grande.
    if tabla.shape[0] <= 25 and tabla.shape[1] <= 25:
        for i in range(tabla.shape[0]):
            for j in range(tabla.shape[1]):
                plt.text(j, i, int(tabla.values[i, j]), ha='center', va='center')
    
    guardar_y_mostrar(titulo)

# Heatmap hora vs estado.
tabla_hora_estado = pd.crosstab(df['hora'], df['estado_seguridad'])
graficar_heatmap(tabla_hora_estado, 'Heatmap de hora vs estado de seguridad', xlabel='Estado', ylabel='Hora')

# Heatmap Event ID vs Logon Type.
tabla_event_logon = pd.crosstab(df['event_id'], df['logon_type'])
graficar_heatmap(tabla_event_logon, 'Heatmap de Event ID vs Logon Type', xlabel='Logon Type', ylabel='Event ID')

# Heatmap host vs Event ID.
tabla_host_event = pd.crosstab(df['host'], df['event_id'])
graficar_heatmap(tabla_host_event, 'Heatmap de Host vs Event ID', xlabel='Event ID', ylabel='Host')

# Heatmap usuario vs Event ID con top 20 usuarios.
tabla_usuario_event = pd.crosstab(df[df['user'].isin(top_usuarios)]['user'],
                                  df[df['user'].isin(top_usuarios)]['event_id'])
graficar_heatmap(tabla_usuario_event, 'Heatmap de Top 20 usuarios vs Event ID', xlabel='Event ID', ylabel='Usuario')


## 7. Análisis específico de eventos sospechosos

In [ ]:
# ============================================================
# 13. ANÁLISIS ESPECÍFICO DE EVENTOS SOSPECHOSOS
# ============================================================

if df_sospechoso.empty:
    print('No existen eventos sospechosos en el dataset.')
else:
    grafico_barras(df_sospechoso['event_id'].value_counts().sort_index(),
                   'Event ID presentes en eventos sospechosos',
                   xlabel='Event ID', rotacion=0)

    grafico_barras(df_sospechoso['user'].value_counts(),
                   'Top usuarios con eventos sospechosos',
                   xlabel='Usuario', rotacion=45, top=20)

    grafico_barras(df_sospechoso['host'].value_counts(),
                   'Hosts con eventos sospechosos',
                   xlabel='Host', rotacion=45)

    grafico_barras(df_sospechoso['domain'].value_counts(),
                   'Dominios asociados a eventos sospechosos',
                   xlabel='Dominio', rotacion=0)

    grafico_barras(df_sospechoso['message'].value_counts(),
                   'Mensajes en eventos sospechosos',
                   xlabel='Mensaje', rotacion=45)

    grafico_barras(df_sospechoso['source_port'].value_counts(),
                   'Top 20 puertos de origen en eventos sospechosos',
                   xlabel='Puerto de origen', rotacion=45, top=20)


## 8. Gráficos de dispersión y boxplot

In [ ]:
# ============================================================
# 14. DISPERSIÓN Y BOXPLOTS
# ============================================================

# Dispersión temporal de eventos por puerto de origen.
plt.figure(figsize=(14, 6))
plt.scatter(df['timestamp'], df['source_port'], alpha=0.5, s=10)
plt.xlabel('Timestamp')
plt.ylabel('Puerto de origen')
guardar_y_mostrar('Dispersión temporal de puertos de origen')

# Dispersión temporal de eventos sospechosos por puerto.
if not df_sospechoso.empty:
    plt.figure(figsize=(14, 6))
    plt.scatter(df_sospechoso['timestamp'], df_sospechoso['source_port'], alpha=0.7, s=15)
    plt.xlabel('Timestamp')
    plt.ylabel('Puerto de origen')
    guardar_y_mostrar('Dispersión temporal de puertos en eventos sospechosos')

# Boxplot de puertos por estado de seguridad.
plt.figure(figsize=(10, 6))
df.boxplot(column='source_port', by='estado_seguridad')
plt.suptitle('')
plt.xlabel('Estado de seguridad')
plt.ylabel('Puerto de origen')
guardar_y_mostrar('Boxplot de puertos por estado de seguridad')

# Boxplot de puertos por Logon Type.
plt.figure(figsize=(12, 6))
df.boxplot(column='source_port', by='logon_type')
plt.suptitle('')
plt.xlabel('Logon Type')
plt.ylabel('Puerto de origen')
guardar_y_mostrar('Boxplot de puertos por Logon Type')


## 9. Resumen ejecutivo automático

In [ ]:
# ============================================================
# 15. RESUMEN EJECUTIVO DEL DATASET
# ============================================================

total_eventos = len(df)
total_sospechosos = int((df['es_sospechoso'] == 1).sum())
total_normales = int((df['es_sospechoso'] == 0).sum())
porcentaje_sospechoso = total_sospechosos / total_eventos * 100 if total_eventos > 0 else 0

print('RESUMEN EJECUTIVO DEL LOG DE SEGURIDAD WINDOWS')
print('=' * 60)
print(f'Total de eventos analizados: {total_eventos:,}')
print(f'Eventos normales: {total_normales:,}')
print(f'Eventos sospechosos: {total_sospechosos:,}')
print(f'Porcentaje de eventos sospechosos: {porcentaje_sospechoso:.2f}%')
print(f'Cantidad de hosts únicos: {df["host"].nunique()}')
print(f'Cantidad de usuarios únicos: {df["user"].nunique()}')
print(f'Cantidad de IPs de origen únicas: {df["source_ip"].nunique()}')
print(f'Cantidad de procesos únicos: {df["process"].nunique()}')
print(f'Rango temporal: {df["timestamp"].min()}  hasta  {df["timestamp"].max()}')

print('\nTop 5 Event ID:')
print(df['event_id'].value_counts().head(5))

print('\nTop 5 usuarios:')
print(df['user'].value_counts().head(5))

print('\nTop 5 IPs de origen:')
print(df['source_ip'].value_counts().head(5))

if not df_sospechoso.empty:
    print('\nTop 5 usuarios con eventos sospechosos:')
    print(df_sospechoso['user'].value_counts().head(5))
    
    print('\nTop 5 IPs con eventos sospechosos:')
    print(df_sospechoso['source_ip'].value_counts().head(5))


## 10. Lista de archivos generados

In [ ]:
# ============================================================
# 16. LISTADO DE GRÁFICOS GUARDADOS
# ============================================================

archivos_generados = sorted(DIR_GRAFICOS.glob('*.png'))

print(f'Se generaron {len(archivos_generados)} gráficos en la carpeta {DIR_GRAFICOS}:')
for archivo in archivos_generados:
    print('-', archivo.name)
